## Use LDA to inspect life history strategy separation for sparse, dense, NSM latents. 
---
Compares three representations of the same specimens side by side: the 28 sparse landmarks, the
dense (population) correspondences, and the NSM latent codes. All three come from the same
`all_vtk_files` order, so one `specimens` table (species / family / trait / color / marker, joined
from `lizard_species_list.csv`) drives every plot.

*Last edited 13 Sep 2026 by K. Wolcott*

In [ ]:
# Imports, paths, and config
import os, re, json, ast, torch
import numpy as np
import pandas as pd
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import make_scorer, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from statsmodels.multivariate.manova import MANOVA

from NSM.plotting import load_mrk_json, plotly_color, n_pcs_for_variance, fmt_p, lda_threshold_grid, manova_stats
from NSM.morphometrics import gm_prcomp

# ---------------------------------------------------------------- TO DO: edit
RUN          = "run_v72"                      # training attempt directory
ATLAS_RUN    = "2026_07-15_13_06_22/"         # atlas/builder run that produced the landmark sets
ATLAS_ROOT = Path("/home/k.wolcott/UFL Dropbox/Katherine Wolcott/neural_shape_models/final_dataset_aug26/atlas/")
CKPT         = "2500"                         # latent code checkpoint to analyze
# -----------------------------------------------------------------------------

cwd       = Path.cwd()
base_wd   = cwd.parent
train_dir = base_wd / RUN
os.chdir(train_dir)
print(f"Working directory: {os.getcwd()}")

ATLAS_DIR       = ATLAS_ROOT / ATLAS_RUN / "atlas"
SPARSE_LM_DIR   = ATLAS_ROOT / ATLAS_RUN / "alignedLMs"
DENSE_LM_DIR    = ATLAS_ROOT / ATLAS_RUN / "population_correspondences"
SPARSE_MEAN_FN  = ATLAS_DIR / "atlas_sparse_landmarks.mrk.json"
DENSE_MEAN_FN   = ATLAS_DIR / "atlas_dense_correspondences.mrk.json"
MEAN_MESH_FN    = ATLAS_DIR / "atlas_model.ply"

OUT_DIR = Path("lda_results")
OUT_DIR.mkdir(exist_ok=True)
print(f"Outputs will be written to: {OUT_DIR.resolve()}")

for p in [ATLAS_DIR, SPARSE_LM_DIR, DENSE_LM_DIR, SPARSE_MEAN_FN, DENSE_MEAN_FN, MEAN_MESH_FN]:
    print(("  OK   " if p.exists() else "  MISS ") + str(p))

# Load config and filenames
with open("model_params_config.json") as f:
    cfg = json.load(f)
print(f"\033[92mLoaded config from model_params_config.json\033[0m")

# Load NSM latent codes
CKPT_PATH = f"latent_codes/{CKPT}.pth"
latent_ckpt = torch.load(CKPT_PATH, map_location="cpu")
codes = latent_ckpt["latent_codes"]["weight"].detach().cpu().numpy()
print(f"Latent codes: {codes.shape}")

# Get filenames
all_vtk_files = [os.path.basename(f) for f in cfg["list_mesh_paths"]]
print(f"{len(all_vtk_files)} meshes listed in config")

# Check filename length against latent codes length
assert len(codes) == len(all_vtk_files), (
    f"{len(codes)} latent codes but {len(all_vtk_files)} meshes in config -- order/count mismatch")

### Specimen metadata (species / family / trait / color / marker)

In [ ]:
# Build specimen metadata table -- one row per mesh, in the same order as `codes`

SPECIES_CSV = "../lizard_species_list.csv"
pat = re.compile(r"^(?P<species>[\w\s\-]+)[\-_ ]+[\w\d]+[\-_ ]+(?P<vertebra>[CTL]?\d+)", re.IGNORECASE)
parsed = [pat.match(f) for f in all_vtk_files]
specimens = pd.DataFrame({"mesh":        all_vtk_files,
                          "specimen_id": [m.group("species") if m else None for m in parsed],
                          "vertebra":    [m.group("vertebra").upper() if m else None for m in parsed]})
print(f"Parsed {specimens['specimen_id'].notna().sum()} / {len(specimens)} filenames")

sdf = pd.read_csv(SPECIES_CSV)
sdf["marker"] = sdf["marker"].astype(str).str.strip().str.strip("'" + chr(34))
sdf["color"]  = sdf["color"].apply(ast.literal_eval)
specimens = specimens.merge(sdf, left_on="specimen_id", right_on="specimen", how="left")

unmatched = specimens.loc[specimens["family"].isna(), "specimen_id"].dropna().unique()
print(f"{specimens['family'].notna().sum()} / {len(specimens)} specimens matched to {SPECIES_CSV}")
if len(unmatched):
    print(f"\033[33mUnmatched specimen IDs ({len(unmatched)}):\033[0m {sorted(unmatched)[:10]}")

REGION_NAMES = {"C": "CERVICAL", "T": "THORACIC", "L": "LUMBAR"}
specimens["region"] = specimens["vertebra"].str[0].map(REGION_NAMES)

print("Specimens dataframe head:\n", specimens.head())

In [ ]:
# Define functions

# One color per trait, derived from that trait's marker 
markers = ['P', '+', 's', 'd', 'X', 'o', '2']
colors = [(0.65, 0.69, 0.12),   # pea soup
        (0.84, 0.65, 0.23),   # saffron
        (0.72, 0.44, 0.22),   # mud
        (0.36, 0.557, 0.68),  # powder blue
        (0.10, 0.51, 0.40),   # deep blue
        (0.60, 0.50, 0.46),   # slate
        (0, 0, 0)]            # black
marker_to_color = dict(zip(markers, colors))

# Match against traits in species_lis.csv
trait_marker = specimens.drop_duplicates("trait").set_index("trait")["marker"]
trait_colors = {t: marker_to_color.get(m, (0.5, 0.5, 0.5))
                for t, m in trait_marker.items() if pd.notna(t)}
specimens["trait_color"] = specimens["trait"].map(trait_colors)
unmapped = [t for t, m in trait_marker.items() if pd.notna(t) and m not in marker_to_color]
if unmapped:
    print(f"\033[33mTraits using a marker not in marker_to_color (defaulting to grey): {unmapped}\033[0m")
n_traits = len(trait_colors)
print(f"{n_traits} traits: {sorted(trait_colors)}")

### Load the three shape representations
Sparse landmarks and dense correspondences are both GPA-aligned/scaled already, so `gm_prcomp`
(Procrustes PCA, from `NSM.morphometrics`) runs directly on them. Latent codes get plain `sklearn`
PCA since they aren't Procrustes shape coordinates.

In [ ]:
# 28 sparse landmarks
sparse_coords = np.stack([load_mrk_json(SPARSE_LM_DIR / (os.path.splitext(f)[0] + ".mrk.json"))[0]
                          for f in all_vtk_files])
sparse_mean, _ = load_mrk_json(SPARSE_MEAN_FN)
print(f"Sparse landmarks: {sparse_coords.shape}")
assert sparse_mean.shape == sparse_coords.shape[1:], "sparse atlas / specimen landmark count mismatch"

pca_sparse = gm_prcomp(sparse_coords)
print(f"{pca_sparse['x'].shape[1]} non-trivial PCs from {sparse_coords.shape[1]*3} sparse coordinates")

# Dense correspondences 
dense_coords = np.stack([load_mrk_json(DENSE_LM_DIR / (os.path.splitext(f)[0] + ".mrk.json"))[0]
                         for f in all_vtk_files])
dense_mean, _ = load_mrk_json(DENSE_MEAN_FN)
print(f"Dense correspondences: {dense_coords.shape}")
assert dense_mean.shape == dense_coords.shape[1:], "dense atlas / specimen point count mismatch"

pca_dense = gm_prcomp(dense_coords)
print(f"{pca_dense['x'].shape[1]} non-trivial PCs from {dense_coords.shape[1]*3} dense coordinates")

# NSM latent codes
pca_latent = PCA(n_components=None).fit(codes)
nsm_latent_scores = pca_latent.transform(codes)
latent_prop = pca_latent.explained_variance_ratio_
latent_cum = np.cumsum(pca_latent.explained_variance_ratio_)
print(f"Latent PCA: {nsm_latent_scores.shape[1]} components, "
      f"PC1+PC2 = {100*(latent_prop[0]+latent_prop[1]):.1f}% of variance")

## LDA / KDE / MANOVA: is group structure meaningful, across representations and label sets?

### Run across all three representations, colored by trait

`N_PC_MATCH` fixes the number of retained PCs the same way across sparse, dense, and latents, so
none of the three gets a separability advantage purely from having more dimensions to work with.

In [ ]:
# Build each representation once at full dimensionality, plus its cumulative-variance 

# Plot params
DPI = 300
PANEL_W_PX, PANEL_H_PX = 700, 650
LDA_FIGSIZE = (PANEL_W_PX / DPI, PANEL_H_PX / DPI)     # (2.333, 2.167) inches
PX_TO_PT = 72 / DPI                                     # CSS px -> matplotlib points
LDA_FONTSIZE      = 30 * PX_TO_PT                       # 7.2 pt  == plotly size=30
LDA_POINT_S       = (6 * PX_TO_PT) ** 2                 # s=2.07  == plotly marker size=6
LDA_LEGEND_MARKER = 20 * PX_TO_PT                       # 4.8 pt  == plotly legend size=20

# PCA dict
reps = {"Landmarks": (pca_sparse["x"], pca_sparse["cum"]),
        "Dense":     (pca_dense["x"],  pca_dense["cum"]),
        "Latents":   (nsm_latent_scores,     latent_cum)}

# PCA retained variance thresholds for inspection
THRESHOLDS = [0.90, 0.95, 0.99, 0.997, 1.0]

curve_rows = []
for target in ["trait", "broad_taxon_for_plotting"]:
    labels = pd.Series(specimens[target].values)
    keep = labels.notna().values
    for name, (X, cum) in reps.items():
        for thr in THRESHOLDS:
            n = n_pcs_for_variance(cum, thr)
            Xs, ys = X[keep, :n], labels[keep].reset_index(drop=True)
            small = ys.value_counts()[lambda c: c < 10].index.tolist()
            k2 = ~ys.isin(small)
            Xs, ys = Xs[k2.values], ys[k2].values
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            scores = cross_val_score(LDA(solver="eigen", shrinkage="auto"), Xs, ys, cv=cv,
                                     scoring=make_scorer(balanced_accuracy_score))
            curve_rows.append({"target": target, "representation": name, "threshold": thr,
                               "n_pcs": n, "accuracy": scores.mean(), "sd": scores.std()})

curve = pd.DataFrame(curve_rows)
for target in curve["target"].unique():
    sub = curve[curve["target"] == target]
    combined = sub.copy()
    combined["value"] = (combined["accuracy"].round(3).astype(str) + " ±" +
                         combined["sd"].round(3).astype(str) + " (n=" +
                         combined["n_pcs"].astype(str) + ")")
    out = (combined.pivot(index="threshold", columns="representation", values="value")
                   .reindex(columns=["Landmarks", "Dense", "Latents"]))
    print(f"\n=== {target} ===")
    print(out.to_string())

### Run across all three representations, colored by broad taxonomic clade

In [ ]:
# LDA 3x3 grid: rows = variance thresholds, cols = representations. One plotly figure,
# same geometry/fonts/marker conventions as the PCA/t-SNE/UMAP panel.

THRESHOLDS = [0.90, 0.95, 0.99]
LABEL_COL, LABEL_TAG = "trait", "trait"
cols = ["Landmarks", "Dense", "Latents"]
col_titles = [f"Landmarks",
              f"Dense correspondences",
              "NSM latents"]

reps_by_threshold = {}
for thr in THRESHOLDS:
    reps_by_threshold[(thr, "Landmarks")] = pca_sparse["x"][:, :n_pcs_for_variance(pca_sparse["cum"], thr)]
    reps_by_threshold[(thr, "Dense")]     = pca_dense["x"][:, :n_pcs_for_variance(pca_dense["cum"], thr)]
    reps_by_threshold[(thr, "Latents")]   = nsm_latent_scores[:, :n_pcs_for_variance(latent_cum, thr)]

_ = lda_threshold_grid(reps_by_threshold, specimens[LABEL_COL].values, trait_colors,
                       THRESHOLDS, cols, col_titles,
                       outstem=f"{RUN}_lda_{LABEL_TAG}_3x3_thresholds", out_dir=OUT_DIR)

## MANOVA

In [ ]:
# MANOVA summary -- 90/95/99% variance thresholds, both label sets, all three representations.
# Provides all numbers needed for the stats one-liner in the methods/results section.

STAT_THRS = [0.90, 0.95, 0.99]

stat_rows = []
for target, label_tag in [("trait", "trait"), ("broad_taxon_for_plotting", "broad taxon")]:
    labels = pd.Series(specimens[target].values)
    keep = labels.notna().values
    for rep_name, (X, cum) in reps.items():
        for thr in STAT_THRS:
            n = n_pcs_for_variance(cum, thr)
            Xs, ys = X[keep, :n], labels[keep].reset_index(drop=True)
            small = ys.value_counts()[lambda c: c < 10].index.tolist()
            k2 = ~ys.isin(small)
            Xs, ys = Xs[k2.values], ys[k2].values
            lev = sorted(set(ys))
            chance = 1 / len(lev)

            lda_fit = LDA(solver="eigen", shrinkage="auto").fit(Xs, ys)
            X_lda   = lda_fit.transform(Xs)
            ms = manova_stats(X_lda, ys)

            # pull CV accuracy already computed in the curve loop above
            cv_row = curve.loc[
                (curve["target"] == target) &
                (curve["representation"] == rep_name) &
                (curve["threshold"] == thr), ["accuracy", "sd"]].iloc[0]

            stat_rows.append({"Label set": label_tag, "Representation": rep_name,
                              "threshold": thr, "n_pcs": n, "n": len(ys),
                              "n_classes": len(lev), "chance": chance,
                              "bal_acc": cv_row["accuracy"], "bal_acc_sd": cv_row["sd"],
                              **ms})

stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(OUT_DIR / f"{RUN}_lda_manova_summary.csv", index=False)

display(stats_df[["Label set", "Representation", "threshold", "n_pcs", "n", "n_classes",
                  "chance", "bal_acc", "bal_acc_sd",
                  "wilks_lambda", "manova_F", "manova_num_df", "manova_den_df",
                  "manova_p", "partial_eta_sq"]].round(4))

for thr in STAT_THRS:
    print(f"\n--- Paper one-liners ({100*thr:.0f}% variance retained) ---")
    for _, r in stats_df[stats_df["threshold"] == thr].iterrows():
        sig = r["manova_p"] < 0.05
        print(f"{r['Label set']} | {r['Representation']} "
                f"({r['n_pcs']} PCs, n={r['n']}, {r['n_classes']} classes, chance={r['chance']:.3f}): "
                f"balanced acc = {r['bal_acc']:.3f} \u00b1 {r['bal_acc_sd']:.3f}, "
                f"MANOVA Wilks\u2019 \u03bb = {r['wilks_lambda']:.4f}, "
                f"F({r['manova_num_df']:.0f}, {r['manova_den_df']:.0f}) = {r['manova_F']:.2f}, "
                f"p = {fmt_p(r['manova_p'], sig)}, "
                f"partial \u03b7\u00b2 = {r['partial_eta_sq']:.3f}")